# BNPL in Europe: Empirical Analysis
**IE University BBA Final Project — Topics in Fintech**  
**Supervisor: Pablo Soler Bach**

*From regulatory arbitrage to regulated credit: how will CCD II reshape BNPL's competitive dynamics in Europe, with evidence from Germany and France?*

---
**Data sources:**
- ECB Statistical Annexes pis2019–pis2025H1 (Table 1 of each annex)
- Worldpay Global Payments Report 2025, p.86
- Klarna Holding AB Annual Report FY2024 + Interim Report H1 2025
- World Bank WDI: NY.GDP.PCAP.CD, IT.NET.USER.ZS, FS.AST.PRVT.GD.ZS, SL.UEM.TOTL.NE.ZS (2023)

**Run order:** Cell 1 (dependencies) → Runtime → Run all

In [ ]:
# Uncomment to install in Google Colab
# !pip install pandas scipy matplotlib numpy -q

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

BLUE   = '#1F4E79'
MID    = '#2E75B6'
GREEN  = '#1E6B3C'
RED    = '#C00000'
ORANGE = '#C55A11'
GRAY   = '#555555'

plt.rcParams.update({
    'font.family':       'serif',
    'font.size':         11,
    'axes.titlesize':    12,
    'axes.titleweight':  'bold',
    'axes.labelsize':    11,
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'figure.dpi':        150,
    'savefig.dpi':       200,
    'savefig.bbox':      'tight',
})

In [ ]:
# ECB Statistical Annexes — Table 1: Relative importance of main payment instruments
# Unit: % of total non-cash transactions by value
# Source: pis2019 through pis2025H1

ecb = pd.DataFrame({
    'Country':   ['Germany','France','Spain','Italy',
                  'Netherlands','Belgium','Finland','Austria','Portugal'],
    'ISO':       ['DE','FR','ES','IT','NL','BE','FI','AT','PT'],
    '2019':      [26.0, 58.6, 60.3, 49.0, 54.2, 50.2, 64.6, 46.7, 69.4],
    '2020':      [28.7, 57.4, 63.4, 47.2, 51.7, 50.7, 63.9, 49.2, 69.6],
    '2021':      [30.3, 59.3, 66.4, 52.5, 49.2, 52.0, 61.6, 51.5, 72.2],
    '2022':      [37.5, 61.2, 66.2, 56.8, 49.2, 58.9, 64.2, 52.9, 74.5],
    '2023-H1':   [40.5, 62.8, 66.6, 56.6, 49.1, 57.0, 64.2, 54.2, 75.5],
    '2023-H2':   [42.1, 63.8, 67.8, 55.3, 49.6, 58.7, 65.8, 56.6, 76.1],
    '2024-H1':   [42.0, 63.5, 66.7, 55.7, 48.8, 58.1, 64.9, 56.8, 75.0],
    '2024-H2':   [43.5, 64.6, 67.6, 56.6, 50.4, 59.1, 64.8, 58.3, 78.6],
    '2025-H1':   [45.3, 63.9, 67.3, 56.6, 48.5, 58.0, 66.1, 57.7, 75.7],
})

periods  = ['2019','2020','2021','2022','2023-H1','2023-H2','2024-H1','2024-H2','2025-H1']
p_labels = ['2019','2020','2021','2022','2023\nH1','2023\nH2','2024\nH1','2024\nH2','2025\nH1']

ecb['delta_pp'] = ecb['2025-H1'] - ecb['2019']
ecb[['Country','2019','2024-H1','2025-H1','delta_pp']]

In [ ]:
# Worldpay Global Payments Report 2025, p.86
# Unit: % of e-commerce transaction value, 2024 (cross-section)

bnpl = pd.DataFrame({
    'Country':  ['Sweden','Germany','Norway','Finland','Belgium',
                 'Denmark','Netherlands','UK','France','Italy',
                 'Ireland','Spain','Poland','Turkey'],
    'ISO':      ['SE','DE','NO','FI','BE','DK','NL','UK',
                 'FR','IT','IE','ES','PL','TR'],
    'BNPL_pct': [23, 20, 18, 16, 12, 11, 10, 7, 5, 5, 3, 3, 3, 1],
    'Bloc':     ['Nordic','Continental','Nordic','Nordic','Continental',
                 'Nordic','Continental','Western','Western','Southern',
                 'Western','Southern','Eastern','Other'],
})
bnpl[['Country','BNPL_pct','Bloc']]

In [ ]:
# World Bank WDI — year 2023 (most recent complete)
# Sweden excluded: non-euro area, no ECB card share data

wb = pd.DataFrame({
    'Country':    ['Germany','France','Netherlands','Spain','Italy'],
    'ISO':        ['DE','FR','NL','ES','IT'],
    'GDP_pc':     [54777, 44700, 63516, 33493, 39277],   # NY.GDP.PCAP.CD
    'Internet':   [92.5,  86.8,  97.0,  95.4,  87.0],   # IT.NET.USER.ZS
    'PrivCredit': [77.3, 112.6,  85.5,  78.4,  63.1],   # FS.AST.PRVT.GD.ZS
    'Unemp':      [3.1,   7.3,   3.5,  12.2,   7.6],    # SL.UEM.TOTL.NE.ZS
    'BNPL_2024':  [20,    5,    10,    3,    5],
    'Color':      [BLUE, ORANGE, GREEN, RED, GRAY],
})
wb[['Country','GDP_pc','Internet','PrivCredit','Unemp','BNPL_2024']]

In [ ]:
# Klarna Holding AB (2025b) Interim Report H1 2025, p.18
klarna_rev = pd.DataFrame({
    'Market':       ['Germany','USA','UK','Sweden','Other'],
    'H1_2024_SEKm': [3095, 3483, 1312, 1161, 2251],
    'H1_2025_SEKm': [3175, 4588, 1674, 1090, 2490],
    'Color':        [BLUE, RED, MID, GREEN, GRAY],
})
klarna_rev['YoY_pct'] = (klarna_rev['H1_2025_SEKm'] / klarna_rev['H1_2024_SEKm'] - 1) * 100

# ECB Statistical Annexes H1+H2 2024, Table 1
ecb_aggregate = {
    'H1_2024_EUR_T': 1.55,
    'H2_2024_EUR_T': 1.72,
    'FY2024_EUR_T':  3.27,
}

# Klarna Annual Report FY2024, p.3 — EUR/USD ~1.09 avg 2024
klarna_gmv_global_usd_b = 105
klarna_pct_ecb = klarna_gmv_global_usd_b / (ecb_aggregate['FY2024_EUR_T'] * 1090) * 100

klarna_rev[['Market','H1_2024_SEKm','H1_2025_SEKm','YoY_pct']]

In [ ]:
# n=5–7: insufficient for formal inference — all results descriptive
# With n=7, |r| > 0.75 required for p < 0.05

cross = pd.DataFrame({
    'Country':    ['Germany','France','Spain','Italy','Netherlands','Belgium','Finland'],
    'Card_2024H1':[42.0, 63.5, 66.7, 55.7, 48.8, 58.1, 64.9],
    'Card_2019':  [26.0, 58.6, 60.3, 49.0, 54.2, 50.2, 64.6],
    'BNPL_2024':  [20, 5, 3, 5, 10, 12, 16],
})
cross['delta_card'] = cross['Card_2024H1'] - cross['Card_2019']

tests = [
    ('Card share H1 2024',       cross['Card_2024H1'], cross['BNPL_2024']),
    ('Card share Δ 2019→2024H1', cross['delta_card'],  cross['BNPL_2024']),
    ('GDP per capita 2023',      wb['GDP_pc'],          wb['BNPL_2024']),
    ('Internet penetration 2023',wb['Internet'],        wb['BNPL_2024']),
    ('Private credit % GDP 2023',wb['PrivCredit'],      wb['BNPL_2024']),
    ('Unemployment 2023',        wb['Unemp'],           wb['BNPL_2024']),
]

corr_results = []
for x_name, x, y in tests:
    r_p, p_p = stats.pearsonr(x, y)
    r_s, p_s = stats.spearmanr(x, y)
    sig = '***' if p_p<0.01 else '**' if p_p<0.05 else '*' if p_p<0.10 else 'n.s.'
    corr_results.append({'Variable': x_name, 'n': len(x),
                         'Pearson_r': r_p, 'p_Pearson': p_p,
                         'Spearman_r': r_s, 'p_Spearman': p_s, 'Sig': sig})

r1,p1 = corr_results[0]['Pearson_r'], corr_results[0]['p_Pearson']
r2,p2 = corr_results[1]['Pearson_r'], corr_results[1]['p_Pearson']
r3,p3 = corr_results[2]['Pearson_r'], corr_results[2]['p_Pearson']
r4,p4 = corr_results[3]['Pearson_r'], corr_results[3]['p_Pearson']
r5,p5 = corr_results[4]['Pearson_r'], corr_results[4]['p_Pearson']
r6,p6 = corr_results[5]['Pearson_r'], corr_results[5]['p_Pearson']

pd.DataFrame(corr_results)[['Variable','n','Pearson_r','p_Pearson','Spearman_r','p_Spearman','Sig']]

In [ ]:
fig1, ax1 = plt.subplots(figsize=(13, 7))

color_map = {
    'Germany': BLUE, 'France': ORANGE, 'Spain': RED,
    'Italy': '#8B4513', 'Netherlands': GREEN,
    'Belgium': '#6B2D8B', 'Finland': GRAY,
    'Austria': '#2ECC71', 'Portugal': '#E74C3C',
}
lw_map = {'Germany': 3.5, 'France': 3.0}

de_vals = [ecb.loc[ecb['Country']=='Germany', p].values[0] for p in periods]
fr_vals = [ecb.loc[ecb['Country']=='France',  p].values[0] for p in periods]

for _, row in ecb.iterrows():
    c    = row['Country']
    vals = [row[p] for p in periods]
    lw   = lw_map.get(c, 1.5)
    ls   = '-' if c in ['Germany','France','Netherlands','Spain'] else '--'
    ax1.plot(p_labels, vals, color=color_map[c], linewidth=lw,
             linestyle=ls, marker='o', markersize=5 if lw > 2 else 3,
             label=c, zorder=5 if lw > 2 else 2)
    ax1.annotate(f"{row['Country']} ({vals[-1]:.1f}%)",
                 xy=(8, vals[-1]), xytext=(8.12, vals[-1]),
                 fontsize=9 if lw < 2 else 10, color=color_map[c],
                 va='center', fontweight='bold' if lw > 2 else 'normal')

ax1.fill_between(range(len(periods)), de_vals, alpha=0.07, color=BLUE)
ax1.fill_between(range(len(periods)), fr_vals, alpha=0.07, color=ORANGE)

ax1.annotate('Germany (Case 1)\n+19.3pp — Gap-filling',
             xy=(8, 45.3), xytext=(6.0, 49), fontsize=9, color=BLUE, ha='center',
             arrowprops=dict(arrowstyle='->', color=BLUE, lw=1.5))
ax1.annotate('France (Case 2)\n+5.3pp — Saturation',
             xy=(8, 63.9), xytext=(6.0, 60), fontsize=9, color=ORANGE, ha='center',
             arrowprops=dict(arrowstyle='->', color=ORANGE, lw=1.5))

ax1.axvline(x=3.5, color='black', linestyle=':', alpha=0.3, lw=1)
ax1.text(3.6, 22, 'Semi-annual\nreporting', fontsize=8, color='gray')
ax1.set_xlabel('Period — Source: ECB Statistical Annexes', fontsize=11)
ax1.set_ylabel('Card payments (% of total non-cash transactions)', fontsize=11)
ax1.set_title(
    'Figure 1: Card Payment Share Evolution, 2019–2025H1\n'
    'Germany (Case 1) vs. France (Case 2) highlighted\n'
    'Source: ECB Statistical Annexes pis2019–pis2025H1', fontsize=12)
ax1.set_ylim(18, 88)
ax1.set_xticks(range(len(periods)))
ax1.set_xticklabels(p_labels)
ax1.grid(axis='y', alpha=0.2, linestyle='--')
plt.tight_layout()
plt.savefig('figure1_card_share_evolution.png')
plt.show()

In [ ]:
fig2, ax2a = plt.subplots(figsize=(12, 6.5))
ax2b = ax2a.twinx()

ax2a.plot(p_labels, de_vals, color=BLUE, linewidth=3.5,
          marker='o', markersize=8, label='Card share % (ECB)', zorder=5)
ax2a.fill_between(range(len(periods)), de_vals, alpha=0.10, color=BLUE)

ax2b.axhline(y=20, color=RED, linewidth=2.5, linestyle='--', zorder=4,
             label='BNPL: 20% e-commerce (Worldpay GPR 2025, p.86)')
ax2b.fill_between(range(len(periods)), 20, alpha=0.05, color=RED)
ax2b.set_ylim(0, 35)
ax2b.set_ylabel('BNPL % of e-commerce (Worldpay GPR 2025)', fontsize=11, color=RED)
ax2b.tick_params(axis='y', labelcolor=RED)

for i, (pl, v) in enumerate(zip(p_labels, de_vals)):
    ax2a.annotate(f'{v:.1f}%', xy=(i, v), xytext=(i, v+1.2),
                  ha='center', fontsize=9, color=BLUE, fontweight='bold')

ax2a.annotate('Fastest-growing card market\nin euro area (+19.3pp)',
              xy=(8, 45.3), xytext=(6.0, 50), fontsize=9.5, color=BLUE, ha='center',
              arrowprops=dict(arrowstyle='->', color=BLUE, lw=1.5))
ax2b.annotate('Highest BNPL in\ncontinental Europe (20%)',
              xy=(4, 20), xytext=(4, 27), fontsize=9.5, color=RED, ha='center',
              arrowprops=dict(arrowstyle='->', color=RED, lw=1.5))

ax2a.set_xlabel('Period', fontsize=11)
ax2a.set_ylabel('Card payments % of non-cash transactions (ECB)', fontsize=11, color=BLUE)
ax2a.tick_params(axis='y', labelcolor=BLUE)
ax2a.set_title(
    'Figure 2: Germany — Card Share vs. BNPL Adoption, 2019–2025H1\n'
    'DUAL GROWTH: both metrics rise simultaneously → Complementarity\n'
    'Sources: ECB Statistical Annexes (card); Worldpay GPR 2025 (BNPL)', fontsize=12)
ax2a.set_xticks(range(len(periods)))
ax2a.set_xticklabels(p_labels)
ax2a.set_ylim(20, 55)
ax2a.grid(axis='y', alpha=0.25)
lines1, lbs1 = ax2a.get_legend_handles_labels()
lines2, lbs2 = ax2b.get_legend_handles_labels()
ax2a.legend(lines1+lines2, lbs1+lbs2, loc='upper left', fontsize=10)
plt.tight_layout()
plt.savefig('figure2_germany_case_study.png')
plt.show()

In [ ]:
fig3, ax3a = plt.subplots(figsize=(12, 6.5))
ax3b = ax3a.twinx()

ax3a.plot(p_labels, fr_vals, color=ORANGE, linewidth=3.5,
          marker='o', markersize=8, label='Card share % (ECB)', zorder=5)
ax3a.fill_between(range(len(periods)), fr_vals, alpha=0.10, color=ORANGE)

ax3b.axhline(y=5, color=RED, linewidth=2.5, linestyle='--', zorder=4,
             label='BNPL: 5% e-commerce (Worldpay GPR 2025, p.86)')
ax3b.fill_between(range(len(periods)), 5, alpha=0.05, color=RED)
ax3b.set_ylim(0, 20)
ax3b.set_ylabel('BNPL % of e-commerce (Worldpay GPR 2025)', fontsize=11, color=RED)
ax3b.tick_params(axis='y', labelcolor=RED)

for i, (pl, v) in enumerate(zip(p_labels, fr_vals)):
    ax3a.annotate(f'{v:.1f}%', xy=(i, v), xytext=(i, v+0.6),
                  ha='center', fontsize=9, color=ORANGE, fontweight='bold')

ax3a.annotate('Card share already high in 2019\n(+5.3pp only — near saturation)',
              xy=(8, 63.9), xytext=(6.0, 60), fontsize=9.5, color=ORANGE, ha='center',
              arrowprops=dict(arrowstyle='->', color=ORANGE, lw=1.5))
ax3b.annotate('BNPL marginal (5%)\ndespite 87% internet penetration',
              xy=(4, 5), xytext=(4, 11), fontsize=9.5, color=RED, ha='center',
              arrowprops=dict(arrowstyle='->', color=RED, lw=1.5))

ax3a.set_xlabel('Period', fontsize=11)
ax3a.set_ylabel('Card payments % of non-cash transactions (ECB)', fontsize=11, color=ORANGE)
ax3a.tick_params(axis='y', labelcolor=ORANGE)
ax3a.set_title(
    'Figure 3: France — Card Share vs. BNPL Adoption, 2019–2025H1\n'
    'HIGH CARD / LOW BNPL: Cartes Bancaires saturation leaves no gap for BNPL\n'
    'Sources: ECB Statistical Annexes (card); Worldpay GPR 2025 (BNPL)', fontsize=12)
ax3a.set_xticks(range(len(periods)))
ax3a.set_xticklabels(p_labels)
ax3a.set_ylim(50, 72)
ax3a.grid(axis='y', alpha=0.25)
lines1, lbs1 = ax3a.get_legend_handles_labels()
lines2, lbs2 = ax3b.get_legend_handles_labels()
ax3a.legend(lines1+lines2, lbs1+lbs2, loc='upper left', fontsize=10)
plt.tight_layout()
plt.savefig('figure3_france_case_study.png')
plt.show()

In [ ]:
fig4, axes4 = plt.subplots(1, 3, figsize=(15, 6))
fig4.suptitle(
    'Figure 4: Germany vs. France — Key Variables Comparison\n'
    'Sources: ECB Statistical Annexes; Worldpay GPR 2025; World Bank WDI (2023)',
    fontsize=12, y=1.01)

ax = axes4[0]
ax.plot(p_labels, de_vals, color=BLUE, linewidth=2.5, marker='o', markersize=5, label='Germany')
ax.plot(p_labels, fr_vals, color=ORANGE, linewidth=2.5, marker='s', markersize=5, label='France')
ax.set_title('Panel A: Card Share Evolution\n(ECB Statistical Annexes)', fontsize=10)
ax.set_ylabel('Card % of non-cash transactions')
ax.set_ylim(20, 75)
ax.set_xticks(range(len(periods)))
ax.set_xticklabels(p_labels, fontsize=7)
ax.legend(fontsize=9)
ax.grid(alpha=0.2)

ax = axes4[1]
ax.bar(['Germany', 'France'], [20, 5], color=[BLUE, ORANGE], width=0.5,
       edgecolor='white', linewidth=1.5)
ax.text(0, 21, '20%', ha='center', fontsize=13, fontweight='bold', color=BLUE)
ax.text(1, 6.2, '5%', ha='center', fontsize=13, fontweight='bold', color=ORANGE)
ax.set_title('Panel B: BNPL % e-commerce 2024\n(Worldpay GPR 2025, p.86)', fontsize=10)
ax.set_ylabel('BNPL % of e-commerce value')
ax.set_ylim(0, 28)
ax.grid(axis='y', alpha=0.2)
ax.annotate('4× difference', xy=(0.5, 12), ha='center', fontsize=10, color='black',
            bbox=dict(boxstyle='round', fc='lightyellow', alpha=0.8))

ax = axes4[2]
categories = ['GDP p.c.\n(USD 000s)', 'Internet\n(%)', 'Private\ncredit (% GDP)', 'Unemploy.\n(%)']
de_vals_wb = [54.8, 92.5, 77.3, 3.1]
fr_vals_wb = [44.7, 86.8, 112.6, 7.3]
x = np.arange(len(categories))
w = 0.35
ax.bar(x - w/2, de_vals_wb, w, label='Germany', color=BLUE, alpha=0.85, edgecolor='white')
ax.bar(x + w/2, fr_vals_wb, w, label='France',  color=ORANGE, alpha=0.85, edgecolor='white')
ax.set_title('Panel C: Macro Controls 2023\n(World Bank WDI)', fontsize=10)
ax.set_xticks(x)
ax.set_xticklabels(categories, fontsize=9)
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.2)
ax.annotate('FR: higher private credit\nbut lower BNPL → ecosystem\nlock-in effect',
            xy=(2+w/2, 112.6), xytext=(2.5, 130), fontsize=7.5, color=ORANGE,
            arrowprops=dict(arrowstyle='->', color=ORANGE, lw=1))

plt.tight_layout()
plt.savefig('figure4_de_fr_comparison.png')
plt.show()

In [ ]:
fig5, axes5 = plt.subplots(2, 2, figsize=(13, 10))
fig5.suptitle(
    'Figure 5: World Bank Macro Controls vs. BNPL Adoption\n'
    'Sources: World Bank WDI (downloaded ZIPs); Worldpay GPR 2025. n=5.',
    fontsize=12, y=1.01)

panels5 = [
    ('GDP_pc',     'GDP per capita 2023 (USD)\n[NY.GDP.PCAP.CD]',
     r3, p3, 'High correlation driven by DE-ES contrast\nKlarna origin-market effect confounds'),
    ('Internet',   'Internet penetration 2023 (%)\n[IT.NET.USER.ZS]',
     r4, p4, 'NL: 97% internet / 10% BNPL → iDEAL\nDigitalisation ≠ BNPL adoption'),
    ('PrivCredit', 'Domestic credit / GDP 2023 (%)\n[FS.AST.PRVT.GD.ZS]',
     r5, p5, 'FR: 112.6% private credit / 5% BNPL\nEcosystem lock-in dominates'),
    ('Unemp',      'Unemployment 2023 (%)\n[SL.UEM.TOTL.NE.ZS]',
     r6, p6, 'Higher unemployment correlates\nwith lower BNPL (risk aversion)'),
]

for ax, (xvar, xlabel, r_val, p_val, note) in zip(axes5.flatten(), panels5):
    for _, row in wb.iterrows():
        ax.scatter(row[xvar], row['BNPL_2024'], color=row['Color'],
                   s=200, zorder=5, edgecolors='white', linewidths=1.5)
        ax.annotate(row['ISO'], xy=(row[xvar], row['BNPL_2024']),
                    xytext=(row[xvar], row['BNPL_2024']+1.2),
                    ha='center', fontsize=11, fontweight='bold', color=row['Color'])
    xv = wb[xvar].values
    yv = wb['BNPL_2024'].values
    m, b = np.polyfit(xv, yv, 1)
    xl = np.linspace(xv.min()*0.96, xv.max()*1.04, 100)
    ax.plot(xl, m*xl+b, GRAY, linestyle='--', alpha=0.6, lw=1.5)
    sig = '***' if p_val<0.01 else '**' if p_val<0.05 else '*' if p_val<0.10 else 'n.s.'
    col = GREEN if abs(r_val)>0.6 else ORANGE if abs(r_val)>0.4 else GRAY
    ax.set_title(f'r = {r_val:+.3f}  p = {p_val:.3f}  {sig}', fontsize=11, color=col)
    ax.set_xlabel(xlabel, fontsize=9)
    ax.set_ylabel('BNPL % e-commerce 2024 (Worldpay)', fontsize=9)
    ax.text(0.03, 0.97, note, transform=ax.transAxes, fontsize=8, color=GRAY, va='top',
            bbox=dict(boxstyle='round', fc='white', alpha=0.7))
    ax.set_ylim(-2, 28)
    ax.grid(alpha=0.2)

plt.tight_layout()
plt.savefig('figure5_macro_controls.png')
plt.show()

In [ ]:
fig6, (ax6a, ax6b) = plt.subplots(1, 2, figsize=(14, 6))
fig6.suptitle(
    'Figure 6: Klarna Scale and Revenue Geography\n'
    'Sources: Klarna Holding AB (2025a) Annual Report FY2024; '
    'Klarna Holding AB (2025b) Interim Report H1 2025; ECB (2025a; 2025b)',
    fontsize=11, y=1.01)

markets = klarna_rev['Market'].tolist()
x6 = np.arange(len(markets))
w6 = 0.38
ax6a.bar(x6-w6/2, klarna_rev['H1_2024_SEKm'], w6, label='H1 2024',
         color=[c+'99' for c in klarna_rev['Color']], edgecolor='white')
ax6a.bar(x6+w6/2, klarna_rev['H1_2025_SEKm'], w6, label='H1 2025',
         color=klarna_rev['Color'], edgecolor='white')
for bar, yoy in zip(ax6a.patches[len(markets):], klarna_rev['YoY_pct']):
    color = GREEN if yoy > 0 else RED
    ax6a.text(bar.get_x()+bar.get_width()/2, bar.get_height()+40,
              f'{yoy:+.0f}%', ha='center', fontsize=9.5, fontweight='bold', color=color)
ax6a.set_xticks(x6)
ax6a.set_xticklabels(markets, fontsize=10)
ax6a.set_ylabel('Net Operating Income (SEK millions)', fontsize=10)
ax6a.set_title('Panel A: Klarna Revenue by Market\n'
               'H1 2024 vs H1 2025 (Klarna H1 2025 Interim Report, p.18)', fontsize=10)
ax6a.legend(fontsize=10)
ax6a.grid(axis='y', alpha=0.25)

entities = ['Klarna GMV\n(Global FY2024)', 'Euro area cards\n(H1 2024)',
            'Euro area cards\n(H2 2024)', 'Euro area cards\n(FY2024)']
values   = [105, 1550, 1720, 3270]
colors6  = [RED, BLUE, BLUE, BLUE]
bars6 = ax6b.bar(entities, values, color=colors6, width=0.55,
                 edgecolor='white', linewidth=1.5, alpha=0.9)
for bar, val, ent in zip(bars6, values, entities):
    unit = 'US$' if 'Klarna' in ent else 'EUR'
    label = f'{unit}{val/1000:.2f}T' if val >= 1000 else f'{unit}{val}B'
    ax6b.text(bar.get_x()+bar.get_width()/2, bar.get_height()+20,
              label, ha='center', va='bottom', fontsize=10, fontweight='bold',
              color=bar.get_facecolor())
ax6b.annotate(
    f'Klarna global = {klarna_pct_ecb:.1f}%\nof euro area cards (FY2024)\n'
    f'(Conservative upper bound)',
    xy=(0, 105), xytext=(1.0, 600), fontsize=9, color=RED, ha='center',
    arrowprops=dict(arrowstyle='->', color=RED, lw=1.5))
ax6b.set_ylabel('Volume (USD/EUR Billions)', fontsize=10)
ax6b.set_title('Panel B: Scale Comparison\nKlarna GMV vs. ECB Euro Area Card Payments 2024',
               fontsize=10)
ax6b.grid(axis='y', alpha=0.25, linestyle='--')

plt.tight_layout()
plt.savefig('figure6_klarna_scale.png')
plt.show()

## Dataset export
Generates the 5 CSV files that accompany the paper. Change `OUTPUT_DIR` to save directly to Google Drive.

In [ ]:
import os

OUTPUT_DIR = '/content/'
# OUTPUT_DIR = '/content/drive/MyDrive/bnpl-ccd2-europe/data/processed/'

ecb_out = ecb.copy()
ecb_out['source']   = 'ECB Statistical Annexes pis2019-pis2025H1'
ecb_out['variable'] = 'card_payments_pct_of_noncash_transactions'
ecb_out['unit']     = '% of total non-cash transactions by value'
ecb_out.to_csv(OUTPUT_DIR + 'dataset1_ecb_card_share.csv', index=False)

bnpl_out = bnpl.copy()
bnpl_out['source']   = 'Worldpay Global Payments Report 2025, p.86'
bnpl_out['variable'] = 'bnpl_pct_ecommerce_value'
bnpl_out['unit']     = '% of e-commerce transaction value, 2024'
bnpl_out.to_csv(OUTPUT_DIR + 'dataset2_bnpl_adoption.csv', index=False)

wb_out = wb[['Country','ISO','GDP_pc','Internet','PrivCredit','Unemp','BNPL_2024']].copy()
wb_out.columns = ['country','iso','gdp_per_capita_usd_2023','internet_penetration_pct_2023',
                  'domestic_credit_pct_gdp_2023','unemployment_pct_2023','bnpl_pct_ecommerce_2024']
wb_out.to_csv(OUTPUT_DIR + 'dataset3_worldbank_macro.csv', index=False)

pd.DataFrame({
    'variable':     ['gdp_per_capita_usd_2023','internet_penetration_pct_2023',
                     'domestic_credit_pct_gdp_2023','unemployment_pct_2023','bnpl_pct_ecommerce_2024'],
    'wb_indicator': ['NY.GDP.PCAP.CD','IT.NET.USER.ZS','FS.AST.PRVT.GD.ZS','SL.UEM.TOTL.NE.ZS','n/a'],
    'year':         [2023, 2023, 2023, 2023, 2024],
    'source':       ['World Bank WDI']*4 + ['Worldpay GPR 2025 p.86'],
}).to_csv(OUTPUT_DIR + 'dataset3_worldbank_macro_sources.csv', index=False)

klarna_out = klarna_rev[['Market','H1_2024_SEKm','H1_2025_SEKm','YoY_pct']].copy()
klarna_out.columns = ['market','h1_2024_sek_m','h1_2025_sek_m','yoy_pct']
klarna_out['source'] = 'Klarna Holding AB (2025b) Interim Report H1 2025, p.18'
klarna_out['unit']   = 'SEK millions, Net Operating Income'
klarna_out.to_csv(OUTPUT_DIR + 'dataset4_klarna_revenue.csv', index=False)

pd.DataFrame({
    'metric': ['klarna_gmv_global_fy2024','ecb_euro_area_cards_h1_2024',
               'ecb_euro_area_cards_h2_2024','ecb_euro_area_cards_fy2024',
               'klarna_global_as_pct_of_euro_area_cards'],
    'value':  [105, 1550, 1720, 3270, round(klarna_pct_ecb, 1)],
    'currency_unit': ['USD billion','EUR billion','EUR billion',
                      'EUR billion (own sum H1+H2)','% (conservative upper bound)'],
    'source': ['Klarna Holding AB (2025a) Annual Report FY2024, p.3',
               'ECB Statistical Annex H1 2024, Table 1',
               'ECB Statistical Annex H2 2024, Table 1',
               'Own calculation from primary ECB data',
               'Own calculation; EUR/USD ~1.09 avg 2024'],
}).to_csv(OUTPUT_DIR + 'dataset5_scale_comparison.csv', index=False)

for f in sorted(os.listdir(OUTPUT_DIR)):
    if f.startswith('dataset'):
        print(f)